# Assignment 5.1
## AAI 540 - MLOps
## University of San Diego, MS in AI


In [20]:
import os

# Set to homework folder
project_dir = "/home/sagemaker-user/aai-540-homework/homework-5-1"
os.chdir(project_dir)
print("Current working directory is now:", os.getcwd())


Current working directory is now: /home/sagemaker-user/aai-540-homework/homework-5-1


## Sagemaker Setup

In [21]:
import boto3
import sagemaker
from sagemaker import image_uris
from datetime import datetime
import os

# Setup SageMaker session and roles
session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
region = session.boto_region_name
s3 = boto3.client("s3")

# Timestamp for unique naming
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

# HW-specific prefix
base_prefix = "hw5-1/lr-monitoring"

# Upload local model artifact to S3
local_model_path = "registry/model.tar.gz"
s3_model_key = f"{base_prefix}/model.tar.gz"
s3.upload_file(local_model_path, bucket, s3_model_key)

# Define URIs for downstream use
model_artifact_uri = f"s3://{bucket}/{s3_model_key}"
batch_output_uri = f"s3://{bucket}/{base_prefix}/batch_output/{timestamp}/"

print("Model Artifact URI:", model_artifact_uri)
print("Batch Output URI:", batch_output_uri)


Model Artifact URI: s3://sagemaker-us-east-1-380537322556/hw5-1/lr-monitoring/model.tar.gz
Batch Output URI: s3://sagemaker-us-east-1-380537322556/hw5-1/lr-monitoring/batch_output/2025-06-23-20-43-24/


In [24]:
# Upload X_val.pkl to S3 for use in monitoring

s3.upload_file("data/X_val.pkl", bucket, "hw5-1/data/X_val.pkl")
print("Uploaded X_val.pkl to s3://{}/hw5-1/data/X_val.pkl".format(bucket))


Uploaded X_val.pkl to s3://sagemaker-us-east-1-380537322556/hw5-1/data/X_val.pkl


## Create Sagemaker Model

In [30]:
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data= model_artifact_uri,  
    role=role,
    entry_point="inference.py",  
    framework_version="1.0-1",   
    sagemaker_session= session
)


## Get Validation Data 

In [31]:
import pickle

# Setup
LOAD_MODE = 'local'  # or 's3'
local_path = "data/X_val.pkl"

# Load locally (applies in both modes)
with open(local_path, "rb") as f:
    X_val = pickle.load(f)

# Optional: reduce size for testing
# X_val = X_val.head(100)

print("Loaded X_val. Shape:", X_val.shape)


Loaded X_val. Shape: (20353, 189)


In [32]:
import os

# Save X_val to CSV without headers or index
os.makedirs("data", exist_ok=True)
csv_path = "data/X_val.csv"
X_val.to_csv(csv_path, index=False, header=False)

# Upload to S3 under hw5-1/batch
batch_input_uri = session.upload_data(
    path=csv_path,
    bucket=bucket,
    key_prefix="hw5-1/batch"
)

print("Uploaded X_val.csv for batch inference.")
print("S3 path:", batch_input_uri)


Uploaded X_val.csv for batch inference.
S3 path: s3://sagemaker-us-east-1-380537322556/hw5-1/batch/X_val.csv


## Batch Inference

In [33]:
import contextlib

transformer = model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    assemble_with="Line",             
    output_path=batch_output_uri,
    accept="text/csv"
)

# Ensure log folder exists
os.makedirs("log", exist_ok=True)

# Run batch transform and suppress streaming logs
with open("log/batch_transform_log.txt", "w") as f, contextlib.redirect_stdout(f):
    transformer.transform(
        data=batch_input_uri,
        content_type="text/csv",
        split_type="Line",
        wait=True
    )

print("Batch transform complete.")
print("Predictions saved to:", batch_output_uri)
print("Logs written to 'batch_transform_log.txt'")


INFO:sagemaker:Creating transform job with name: sagemaker-scikit-learn-2025-06-23-21-11-49-304


Batch transform complete.
Predictions saved to: s3://sagemaker-us-east-1-380537322556/hw5-1/lr-monitoring/batch_output/2025-06-23-20-43-24/
Logs written to 'batch_transform_log.txt'


## Inspect Predictions

In [34]:
import re
import pandas as pd

def download_batch_output(s3_uri):
    match = re.match(r"s3://([^/]+)/(.+)", s3_uri)
    bucket_name, prefix = match.group(1), match.group(2)

    s3 = boto3.client("s3")
    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
    output_file = next(obj["Key"] for obj in response["Contents"] if obj["Key"].endswith(".out"))

    s3.download_file(bucket_name, output_file, "log/predictions.out")
    return pd.read_csv("log/predictions.out", header=None)

df_preds = download_batch_output(batch_output_uri)


In [35]:
# Compare results

# Path setup
local_path = "data/y_val.pkl"
s3_key = "hw5-1/data/y_val.pkl"

LOAD_MODE = 'local'

if LOAD_MODE == "s3":
    print("Loading X_val from S3...")
    s3 = boto3.client("s3")
    os.makedirs("data", exist_ok=True)  # Ensure local folder exists
    with open(local_path, "wb") as f:
        s3.download_fileobj(bucket, s3_key, f)

# Load from local file (works for both local and S3 modes)
with open(local_path, "rb") as f:
    y_val = pickle.load(f)

# Create smaller file for faster execution
# y_val = y_val.head(100)
# print("Loaded y_val. Shape:", y_val.shape)

# Reset index for proper alignment
y_val = y_val.reset_index(drop=True)
df_preds = df_preds.reset_index(drop=True)
df_preds.columns = ["y_pred"]

df_eval = pd.DataFrame({
    "y_true": y_val,
    "y_pred": df_preds["y_pred"]
})

# Ensure index alignment
race_col = X_val["race_AfricanAmerican"].reset_index(drop=True)

# Add to df_eval
df_eval["race_AfricanAmerican"] = race_col

# Preview
df_eval.head()


,y_true,y_pred,race_AfricanAmerican
0,0,0,False
1,1,0,True
2,0,1,False
3,0,0,True
4,0,0,False


In [36]:
# Classification report on validation data

from sklearn.metrics import classification_report

print(classification_report(df_eval["y_true"], df_eval["y_pred"]))


              precision    recall  f1-score   support

           0       0.92      0.66      0.77     18082
           1       0.17      0.54      0.25      2271

    accuracy                           0.65     20353
   macro avg       0.54      0.60      0.51     20353
weighted avg       0.84      0.65      0.71     20353



## Output Batch Inference

In [37]:
import os
import boto3

# Save df_eval as inference.csv
local_target_dir = "data"
os.makedirs(local_target_dir, exist_ok=True)

local_target_path = os.path.join(local_target_dir, "inference.csv")
df_eval.to_csv(local_target_path, index=False)

print(f"Saved df_eval to: {local_target_path}")

# Upload to S3
s3 = boto3.client("s3")
s3_target_key = "hw5-1/batch_output/inference.csv"

s3.upload_file(local_target_path, bucket, s3_target_key)
print(f"Uploaded to S3 at: s3://{bucket}/{s3_target_key}")


Saved df_eval to: data/inference.csv
Uploaded to S3 at: s3://sagemaker-us-east-1-380537322556/hw5-1/batch_output/inference.csv


# Bias Monitor

In [38]:
# Save the 3-column df_eval for bias monitoring
bias_path = "data/bias_baseline.csv"
df_eval.to_csv(bias_path, index=False)

# Upload to S3
bias_s3_key = "hw5-1/bias/baseline.csv"
s3.upload_file(bias_path, bucket, bias_s3_key)

# Store S3 URI for later use
bias_baseline_s3_uri = f"s3://{bucket}/{bias_s3_key}"
print("Uploaded bias baseline to:", bias_baseline_s3_uri)


Uploaded bias baseline to: s3://sagemaker-us-east-1-380537322556/hw5-1/bias/baseline.csv


In [47]:
from sagemaker.model_monitor.clarify_model_monitoring import ModelBiasMonitor
from sagemaker.clarify import DataConfig, BiasConfig, ModelPredictedLabelConfig

# Data Config
data_config = DataConfig(
    s3_data_input_path=bias_baseline_s3_uri,
    s3_output_path=f"s3://{bucket}/hw5-1/bias/baseline_output",  
    label="y_true",
    predicted_label="y_pred",
    headers=["y_true", "y_pred", "race_AfricanAmerican"],
    dataset_type="text/csv"
)

# BiasConfig
bias_config = BiasConfig(
    label_values_or_threshold=[1],
    facet_name="race_AfricanAmerican",
    facet_values_or_threshold=[1]
)

# Predicted-label config for batch transform outputs  
model_predicted_label_config = ModelPredictedLabelConfig(
    label="y_pred"
)


In [48]:
# Create bias monitor instance
bias_monitor = ModelBiasMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=session
)

# Run bias baseline without endpoint, since y_pred is already in the dataset
bias_monitor.suggest_baseline(
    data_config=data_config,       # Includes y_true, y_pred, race_AfricanAmerican
    bias_config=bias_config,       # Defines the facet group and label class
    model_config=None,             # No deployed endpoint needed
    wait=True,
    logs=True
)


INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker.clarify:Analysis Config: {'dataset_type': 'text/csv', 'headers': ['y_true', 'y_pred', 'race_AfricanAmerican'], 'label': 'y_true', 'predicted_label': 'y_pred', 'label_values_or_threshold': [1], 'facet': [{'name_or_index': 'race_AfricanAmerican', 'value_or_threshold': [1]}], 'methods': {'report': {'name': 'report', 'title': 'Analysis Report'}, 'pre_training_bias': {'methods': 'all'}, 'post_training_bias': {'methods': 'all'}}}
INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2025-06-23-22-30-03-614


...................sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml
We are not in a supported iso region, /bin/sh exiting gracefully with no changes.
INFO:sagemaker-clarify-processing:Starting SageMaker Clarify Processing job
INFO:analyzer.data_loading.data_loader_util:Analysis config path: /opt/ml/processing/input/config/analysis_config.json
INFO:analyzer.data_loading.data_loader_util:Analysis result path: /opt/ml/processing/output
INFO:analyzer.data_loading.data_loader_util:This host is algo-1.
INFO:analyzer.data_loading.data_loader_util:This host is the leader.
INFO:analyzer.data_loading.data_loader_util:Number of hosts in the cluster is 1.
INFO:sagemaker-clarify-processing:Running Python / Pandas based analyzer.
INFO:analyzer.data_loading.data_loader_factory:Dataset type: text/csv uri: /opt/ml/processing/input/data
INFO:sagemaker-clarif

In [50]:
# List S3 output files from the bias baseline job
files = s3.list_objects(Bucket=bucket, Prefix="hw5-1/bias/baseline_output")

if "Contents" in files:
    output_files = [obj["Key"] for obj in files["Contents"]]
    print("Files in S3 'baseline_output' folder:")
    for f in output_files:
        print("-", f)
else:
    print("No output files found.")


Files in S3 'baseline_output' folder:
- hw5-1/bias/baseline_output//analysis_config.json
- hw5-1/bias/baseline_output/analysis.json
- hw5-1/bias/baseline_output/analysis_config.json
- hw5-1/bias/baseline_output/report.html
- hw5-1/bias/baseline_output/report.ipynb
- hw5-1/bias/baseline_output/report.pdf


In [52]:
# Render HTML report inline
from IPython.display import IFrame

# Download report.html
s3.download_file(bucket, "hw5-1/bias/baseline_output/report.html", "report.html")

# Display inline in notebook
IFrame(src="report.html", width=900, height=600)


## Bias Monitoring Summary

I conducted post-training bias analysis using SageMaker Clarify’s ModelBiasMonitor to evaluate the fairness of our logistic regression model with respect to the feature race_AfricanAmerican.

The input to the monitor included:

y_true: actual readmission outcomes

y_pred: model predictions

race_AfricanAmerican: binary indicator of racial group

Key findings from the analysis:

Disparate Impact (DI) was approximately 1.01, indicating nearly equal rates of positive predictions across groups.

Recall Difference (RD) and Difference in Positive Proportions (DPPL) were close to zero, suggesting minimal disparity in performance.

Several fairness metrics such as Treatment Equality, Conditional Acceptance, and Specificity Difference confirmed consistent model behavior across racial groups.

Some advanced metrics such as Flip Test (FT) and CDDPL failed due to missing feature data required for those tests. These are not critical for a high-level fairness assessment.

In summary, the bias monitor showed no major evidence of racial bias in model predictions, indicating the model behaves fairly on this axis in our validation data.